In [3]:
#find local directory
getwd()

#set working directory
setwd("/home/user/EDS_Capstone_Project/US_Renewable_Energy_EDS_Capstone/US_Renewable_Energy_EDS_Capstone/Lesson_1:Cleaning_Data_for_Analysis")
getwd()

#set main file path
file_path <- '/home/user/EDS_Capstone_Project/US_Renewable_Energy_EDS_Capstone/US_Renewable_Energy_EDS_Capstone'
file_path

[1] "/home/user/EDS_Capstone_Project/US_Renewable_Energy_EDS_Capstone/US_Renewable_Energy_EDS_Capstone/Lesson_1:Cleaning_Data_for_Analysis/cleaned_datasets_scripts"

[1] "/home/user/EDS_Capstone_Project/US_Renewable_Energy_EDS_Capstone/US_Renewable_Energy_EDS_Capstone/Lesson_1:Cleaning_Data_for_Analysis"

[1] "/home/user/EDS_Capstone_Project/US_Renewable_Energy_EDS_Capstone/US_Renewable_Energy_EDS_Capstone"

In [4]:
#load in packages
library(tidyverse)

#load in raw data
total_energy <- read.csv(file.path(file_path, "raw_data/Total_State_Energy_Consumption_(billion_BTU).csv"))
renewable_energy <- read.csv(file.path(file_path, "raw_data/State_Renewable_Energy_Consumption_(billion_BTU).csv"))

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     


── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [5]:
#look at data
#head(total_energy)
head(renewable_energy)

#want years to be a variable column (Year)instead of on top
#want to change years from X####-> ####
#must delete "," in numbers

,State,X1960,X1961,X1962,X1963,X1964,X1965,X1966,X1967,X1968,⋯,X2014,X2015,X2016,X2017,X2018,X2019,X2020,X2021,X2022,X2023
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,AK,"4,670","5,160","5,283","5,493","5,828","6,057","6,114","5,910","5,946",⋯,"11,523","13,526","14,371","13,136","13,694","12,952","9,757","9,597","10,410","10,087"
2,AL,"66,967","69,190","71,684","67,547","76,231","71,876","72,734","80,231","78,106",⋯,"236,143","225,388","217,520","229,887","233,969","232,118","235,538","239,817","232,035","222,189"
3,AR,"40,817","41,049","40,059","38,411","38,775","38,819","40,584","37,500","43,993",⋯,"115,127","107,425","105,871","103,841","105,007","106,732","91,389","89,714","90,825","87,277"
4,AZ,"14,214","13,847","13,645","14,064","13,932","18,841","21,495","21,267","23,609",⋯,"72,405","79,520","83,630","88,295","92,790","92,495","90,813","99,266","101,215","108,445"
5,CA,"141,733","136,263","165,152","180,034","174,530","202,276","190,831","223,328","204,480",⋯,"560,635","564,184","645,141","724,945","690,792","788,792","738,585","810,020","880,995","1,065,179"
6,CO,"9,788","9,415","9,886","10,099","10,420","9,768","10,388","10,391","10,654",⋯,"71,727","76,060","86,993","88,198","90,564","98,087","95,342","103,955","114,917","115,061"


In [6]:
#PIVOT LONGER to get duplicate states
#want State in first column, Year in second column, energy consumption in third row

total_energy_long <- total_energy %>%
  pivot_longer(
    cols = 2:65, #duplicates the state and keeps it in the first column
    names_to = "Year", #variable for headings
    values_to = "Total_Energy" #variables for observations
  ) %>%
  mutate(Year = str_replace(Year, "[X]", "")) %>% #deletes X in front of year
  mutate(Total_Energy = as.numeric(str_replace_all(Total_Energy, ",", ""))) #char -> num and deletes ","s in number
    
#total_energy_long

renewable_energy_long <- renewable_energy %>%
  pivot_longer(
    cols = 2:65, #duplicates the state and keeps it in the first column
    names_to = "Year", #variable for headings
    values_to = "Renewable_Energy" #variables for observations
  ) %>%
  mutate(Year = str_replace(Year, "[X]", "")) %>% #deletes X in front of year
  mutate(Renewable_Energy = as.numeric(str_replace_all(Renewable_Energy, ",", ""))) #char -> num & deletes ","s in number
    
head(renewable_energy_long)

State,Year,Renewable_Energy
<chr>,<chr>,<dbl>
AK,1960,4670
AK,1961,5160
AK,1962,5283
AK,1963,5493
AK,1964,5828
AK,1965,6057


In [8]:
#join the two datasets together by State and Year

energy_consumption <- total_energy_long %>%
    full_join(renewable_energy_long, by = c("State", "Year")) %>%
    mutate(NonRenewable_Energy = Total_Energy - Renewable_Energy) %>%

    #percentages
    mutate(NonRenewable_Percent = round((NonRenewable_Energy/Total_Energy)*100)) %>%    
    mutate(Renewable_Percent = round((Renewable_Energy/Total_Energy)*100)) %>% 

    #ordered
    select(State, Year, Total_Energy, NonRenewable_Energy, NonRenewable_Percent, Renewable_Energy, Renewable_Percent)

energy_consumption
#WV and WY are there!!

State,Year,Total_Energy,NonRenewable_Energy,NonRenewable_Percent,Renewable_Energy,Renewable_Percent
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
AK,1960,59303,54633,92,4670,8
AK,1961,70020,64860,93,5160,7
AK,1962,76642,71359,93,5283,7
AK,1963,78488,72995,93,5493,7
AK,1964,82793,76965,93,5828,7
AK,1965,85319,79262,93,6057,7
AK,1966,100481,94367,94,6114,6
AK,1967,112625,106715,95,5910,5
AK,1968,119992,114046,95,5946,5


In [9]:
write.csv(energy_consumption, file.path(file_path, "Lesson_1:Cleaning_Data_for_Analysis/cleaned_datasets/State_Energy_Consumptions.csv"))